# Seminar 9

Few free requests

In [ ]:
!pip install gigachain_community python_dotenv --quiet

Prepare LLM

`profanity_check` - do we check the content

In [ ]:
#from langchain.prompts import load_prompt
#from langchain.chains.summarize import load_summarize_chain
#from langchain_community.chat_models import GigaChat


giga = GigaChat(credentials=TOKEN,
                model="GigaChat-Pro")

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


loader = TextLoader("c_a_p.txt")
documents = loader.load()

#Divide by chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 7000,
    chunk_overlap  = 0,
    length_function = len,
    is_separator_regex = False,
)
documents = text_splitter.split_documents(documents)
print(f"Number of chunks: {len(documents)}")

Number of chunks: 12


In [ ]:
documents[9].page_content[-100:]

'o. The drunken man was more and more overcome by dismay and\nconfusion as they drew nearer the house.'

Prompts may be written in text or saved as the configuration:

In [ ]:
%%writefile summarize_book_map.yaml
input_variables: [text]
partial_variables:
    map_size: '2-3 sentences'
template: 'Write summarization of part of the book in {map_size}.


"{text}"


Write summarization of the book in {map_size}:'
template_format: f-string
_type: prompt

Overwriting summarize_book_map.yaml


In [ ]:
%%writefile summarize_book_combine.yaml
input_variables: [text]
partial_variables:
    combine_size: '10-20 sentences'
template: 'Write summarization of part of the book in {combine_size}.


"{text}"


Write summarization of part of the book in {combine_size}.
Summarization:'
template_format: f-string
_type: prompt

Overwriting summarize_book_combine.yaml


Upload prompts:

In [ ]:
book_map_prompt = load_prompt("summarize_book_map.yaml")
book_combine_prompt = load_prompt("summarize_book_combine.yaml")

In [ ]:
book_combine_prompt

PromptTemplate(input_variables=['text'], partial_variables={'combine_size': '10-20 sentences'}, template='Write summarization of part of the book in {combine_size}.\n\n"{text}"\n\nWrite summarization of part of the book in {combine_size}. Summarization:')

Prepare for summarizing:

In [ ]:
chain = load_summarize_chain(giga, chain_type="map_reduce",
                             map_prompt=book_map_prompt,
                             combine_prompt=book_combine_prompt,
                             verbose=False)

Go

In [ ]:
res = chain.invoke({"input_documents": documents})

ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1006)

The result:

In [ ]:
print(res["output_text"].replace(". ", ".\n"))

TypeError: 'ChatCompletion' object is not subscriptable

#Vector database qdrant

Database with wine.

In [ ]:
import pandas as pd
df = pd.read_csv('top_rated_wines.csv')
df = df[df['variety'].notna()]
data = df.sample(700).to_dict('records')
len(data)

700

In [ ]:
df.sample(5)

,name,region,variety,rating,notes
539,Chateau Leoville Barton (Futures Pre-Sale) 2019,"St-Julien, Bordeaux, France",Red Wine,96.0,"Black cherry nose, blackberry, touch of mocha...."
734,Clos Mogador Priorat 2001,"Priorat, Spain",Red Wine,97.0,"The wine shows an opaque black, purple colour ..."
398,Chateau de Beaucastel Hommage Jacques Perrin C...,"Chateauneuf-du-Pape, Rhone, France",Red Wine,96.0,The Chateau de Beaucastel Hommage Jacques Perr...
171,Bodegas El Nido El Nido 2006,"Jumilla, Spain",Red Wine,97.0,"This red is jam packed with blackberry, black ..."
16,Agharta Black Label Red 2004,"North Coast, California",Red Wine,98.0,"Evolved aromas of roasted earth, dried leather..."


In [ ]:
df['notes'][0]

'Vintage Comments : Classic Barossa vintage conditions. An average wet Spring followed by extreme heat in early February. Occasional rainfall events kept the vines in good balance up to harvest in late March 2004. Very good quality coupled with good average yields. More than 30 months in wood followed by six months tank maturation of the blend prior to bottling, July 2007. '

Qdrant:
https://github.com/qdrant/qdrant-client


Embedding model: 'all-MiniLM-L6-v2' from sentence_transformers

In [ ]:
!pip install qdrant_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.7/306.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 21.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.6
    Uninstalling protobuf-4.25.6:
      Successfully uninstalled protobuf-4.25.6
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.70.0
    Uninstalling grpcio-1.70.0:
      Successfully uninstalled grpcio-1.70.0


In [ ]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
encoder.encode(['This framework generates embeddings for each input sentence']).shape

(1, 384)

In [ ]:
#Create BD in memory
qdrant = QdrantClient(":memory:")

In [ ]:
encoder.get_sentence_embedding_dimension()

384

Create collection:

In [ ]:
qdrant.recreate_collection(
    collection_name="top_wines",
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(),
        distance=models.Distance.COSINE
    )
)

<ipython-input-55-903da2cf74ec>:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

Vectorize:

In [ ]:
qdrant.upload_points(
    collection_name="top_wines",
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(doc["notes"]).tolist(),
            payload=doc,
        ) for idx, doc in enumerate(data)
    ]
)

In [ ]:
for idx, doc in enumerate(data):
  print(doc)
  break

{'name': 'Booker Vineyard Ripper Grenache 2011', 'region': 'Paso Robles, Central Coast, California', 'variety': 'Red Wine', 'rating': 96.0, 'notes': 'Most consider our Ripper to be the most elegant wine we produce with its delicate red fruits and hints of cherry cola. We partially age the wine in neutral French 500L oak barrels, and the other part of the wine in concrete tanks. The Ripper is only made in select years when our Grenache crop shows top-quality.'}


Request:

In [ ]:
prompt = "Suggest me an amazing Malbec wine from Argentina"

By hand:

In [ ]:
df[(df['name'].apply(lambda x: 'Malbec' in x)) & (df['region'].apply(lambda x: 'Argentina' in x))].sort_values('rating', ascending=False)[:5]

,name,region,variety,rating,notes
292,Catena Zapata Adrianna Vineyard River Stones M...,"Uco Valley, Mendoza, Argentina",Red Wine,98.0,This cuvée takes its name from a small parcel ...
296,Catena Zapata Argentino Vineyard Malbec 2004,Argentina,Red Wine,98.0,"""The single-vineyard 2004 Malbec Argentino Vin..."
291,Catena Zapata Adrianna Vineyard Malbec 2004,Argentina,Red Wine,97.0,"""The single-vineyard 2004 Malbec Adrianna Vine..."
43,Altos las Hormigas Gualtallary Malbec 2016,"Uco Valley, Mendoza, Argentina",Red Wine,96.0,"A solid violet red color reveals a dense, conc..."
44,Altos las Hormigas Gualtallary Malbec 2017,"Uco Valley, Mendoza, Argentina",Red Wine,96.0,"A solid violet red color reveals a dense, conc..."


Search in database:

In [ ]:
hits = qdrant.search(
    collection_name="top_wines",
    query_vector=encoder.encode(prompt).tolist(),
    limit=5
)
for hit in hits:
  print(hit.payload, "score:", hit.score)

{'name': 'Catena Zapata Argentino Vineyard Malbec 2004', 'region': 'Argentina', 'variety': 'Red Wine', 'rating': 98.0, 'notes': '"The single-vineyard 2004 Malbec Argentino Vineyard spent 17 months in new French oak. Remarkably fragrant and complex aromatically, it offers up aromas of wood smoke, creosote, pepper, clove, black cherry, and blackberry. Made in a similar, elegant style, it is the most structured of the three single vineyard wines, needing a minimum of a decade of additional cellaring. It should easily prove to be a 25-40 year wine. It is an exceptional achievement in Malbec. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge i

<ipython-input-64-5ef5714fedea>:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(


In [ ]:
hits[0]

ScoredPoint(id=524, version=0, score=0.6377782347562875, payload={'name': 'Catena Zapata Argentino Vineyard Malbec 2004', 'region': 'Argentina', 'variety': 'Red Wine', 'rating': 98.0, 'notes': '"The single-vineyard 2004 Malbec Argentino Vineyard spent 17 months in new French oak. Remarkably fragrant and complex aromatically, it offers up aromas of wood smoke, creosote, pepper, clove, black cherry, and blackberry. Made in a similar, elegant style, it is the most structured of the three single vineyard wines, needing a minimum of a decade of additional cellaring. It should easily prove to be a 25-40 year wine. It is an exceptional achievement in Malbec. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architect

In [ ]:
search_results = [hit.payload for hit in hits]

In [ ]:
search_results

[{'name': 'Catena Zapata Argentino Vineyard Malbec 2004',
  'region': 'Argentina',
  'variety': 'Red Wine',
  'rating': 98.0,
  'notes': '"The single-vineyard 2004 Malbec Argentino Vineyard spent 17 months in new French oak. Remarkably fragrant and complex aromatically, it offers up aromas of wood smoke, creosote, pepper, clove, black cherry, and blackberry. Made in a similar, elegant style, it is the most structured of the three single vineyard wines, needing a minimum of a decade of additional cellaring. It should easily prove to be a 25-40 year wine. It is an exceptional achievement in Malbec. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given th

In [ ]:
df['rating'].mean()

96.85894580549369

Possible problems:

In [ ]:
hits = qdrant.search(
    collection_name="top_wines",
    query_vector=encoder.encode('White wine from France').tolist(),
    limit=5
)
for hit in hits:
  print(hit.payload, "score:", hit.score)

{'name': 'Gemstone Vineyard Estate Cabernet Sauvignon 2013', 'region': 'Yountville, Napa Valley, California', 'variety': 'Red Wine', 'rating': 96.0, 'notes': 'This is a well-structured Cabernet Sauvignon, with notes of crème de cassis, rose, black chocolate, baked fruit, tobacco and vanilla bean. We love it for it’s juicy mouth feel, layered textured and a lingering finish with just the right amount of acidity to balance the mature tannins. Produced from 6 different blocks on the estate, this wine is a true reflection of the diversity of our property, in addition to an exceptional vintage with long aging potential. '} score: 0.593188186646646
{'name': 'Chateau de Saint Cosme Gigondas le Poste 2010', 'region': 'Gigondas, Rhone, France', 'variety': 'Red Wine', 'rating': 98.0, 'notes': "Le Poste is a favourite wine at Saint Cosme. It is everything they like in wine - it is a wine of fruit and a wine to keep, a wine of complexity and an easy to understand one, a wine of structure and a win

<ipython-input-69-4971eeaf689f>:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(


# Another vector database

Chroma: https://github.com/chroma-core/chroma

In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.8 MB/s eta 0:0

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

CHROMA_DATA_PATH = "chroma_db/"
COLLECTION_NAME = "my_first_collection" #Name of our collection

client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)

Embedding model + metric

In [ ]:
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
 name=COLLECTION_NAME,
embedding_function=embedding_func,
 metadata={"hnsw:space": "cosine"},
)

Example using the lists:

In [ ]:
documents = [
    "The latest iPhone model comes with impressive features and a powerful camera.",
    "Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.",
    "Einstein's theory of relativity revolutionized our understanding of space and time.",
    "Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.",
    "The American Revolution had a profound impact on the birth of the United States as a nation.",
    "Regular exercise and a balanced diet are essential for maintaining good physical health.",
    "Leonardo da Vinci's Mona Lisa is considered one of the most iconic paintings in art history.",
    "Climate change poses a significant threat to the planet's ecosystems and biodiversity.",
    "Startup companies often face challenges in securing funding and scaling their operations.",
    "Beethoven's Symphony No. 9 is celebrated for its powerful choral finale, 'Ode to Joy.'",
]

genres = [
    "technology",
    "travel",
    "science",
    "food",
    "history",
    "fitness",
    "art",
    "climate change",
    "business",
    "music",
]

collection.add(
    documents=documents,
    ids=[f"id{i}" for i in range(len(documents))],
    metadatas=[{"genre": g} for g in genres]
)

Request + number of results we need

In [ ]:
collection.query(
    query_texts=["I want to eat!!!11"],
    n_results=7,
)

{'ids': [['id5', 'id3', 'id1', 'id9', 'id2', 'id7', 'id0']],
 'embeddings': None,
 'documents': [['Regular exercise and a balanced diet are essential for maintaining good physical health.',
   'Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.',
   'Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.',
   "Beethoven's Symphony No. 9 is celebrated for its powerful choral finale, 'Ode to Joy.'",
   "Einstein's theory of relativity revolutionized our understanding of space and time.",
   "Climate change poses a significant threat to the planet's ecosystems and biodiversity.",
   'The latest iPhone model comes with impressive features and a powerful camera.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'genre': 'fitness'},
   {'genre': 'food'},
   {'genre': 'travel'},
   {'genre': 'music'},
   {'genre': 'science'},
   {'genre': 'climate change'},
   {'genre': 'technology'}]],
 'distances': [[0.783552

In [ ]:
collection.query(
    query_texts=["Name some composer"],
    n_results=5,
)

{'ids': [['id9', 'id6', 'id3', 'id2', 'id7']],
 'embeddings': None,
 'documents': [["Beethoven's Symphony No. 9 is celebrated for its powerful choral finale, 'Ode to Joy.'",
   "Leonardo da Vinci's Mona Lisa is considered one of the most iconic paintings in art history.",
   'Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.',
   "Einstein's theory of relativity revolutionized our understanding of space and time.",
   "Climate change poses a significant threat to the planet's ecosystems and biodiversity."]],
 'uris': None,
 'data': None,
 'metadatas': [[{'genre': 'music'},
   {'genre': 'art'},
   {'genre': 'food'},
   {'genre': 'science'},
   {'genre': 'climate change'}]],
 'distances': [[0.6753662601628385,
   0.8039844728059002,
   0.8596070153349942,
   0.8841103183789923,
   0.9571625502262026]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [ ]:
collection.query(
    query_texts=["I want to eat"],
    n_results=3,
)

{'ids': [['id5', 'id1', 'id3']],
 'embeddings': None,
 'documents': [['Regular exercise and a balanced diet are essential for maintaining good physical health.',
   'Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.',
   'Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'genre': 'fitness'}, {'genre': 'travel'}, {'genre': 'food'}]],
 'distances': [[0.7541260614541149, 0.8258124500924363, 0.8593753824601559]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [ ]:
collection.query(
    query_texts=["Teach me about history",
                 "What's going on in the world?"],
    n_results=3
)

{'ids': [['id2', 'id4', 'id6'], ['id7', 'id2', 'id1']],
 'embeddings': None,
 'documents': [["Einstein's theory of relativity revolutionized our understanding of space and time.",
   'The American Revolution had a profound impact on the birth of the United States as a nation.',
   "Leonardo da Vinci's Mona Lisa is considered one of the most iconic paintings in art history."],
  ["Climate change poses a significant threat to the planet's ecosystems and biodiversity.",
   "Einstein's theory of relativity revolutionized our understanding of space and time.",
   'Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'genre': 'science'}, {'genre': 'history'}, {'genre': 'art'}],
  [{'genre': 'climate change'}, {'genre': 'science'}, {'genre': 'travel'}]],
 'distances': [[0.6265882477735377, 0.6904193021456559, 0.8771599724748073],
  [0.8002943757342755, 0.8882107082405322, 0.8892552337366297]],
 'included'

In [ ]:
query = "I am hungry"
res = collection.query(
    query_texts=[query],
    n_results=3,
)
res

{'ids': [['id3', 'id1', 'id5']],
 'embeddings': None,
 'documents': [['Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.',
   'Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.',
   'Regular exercise and a balanced diet are essential for maintaining good physical health.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'genre': 'food'}, {'genre': 'travel'}, {'genre': 'fitness'}]],
 'distances': [[0.8819609740917329, 0.9224464867282016, 0.9345032689776885]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

# RAG

Now, we can create new prompt using the extracted documents:

In [ ]:
relevant_docs = '\n//\n'.join(res['documents'][0])

print(relevant_docs)

Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.
//
Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.
//
Regular exercise and a balanced diet are essential for maintaining good physical health.


In [ ]:
new_prompt = ('Here are some documents, separated by // :\n' + relevant_docs +
              '\nHelp user with the following request: ' + query)

print(new_prompt)

Here are some documents, separated by // :
Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.
//
Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.
//
Regular exercise and a balanced diet are essential for maintaining good physical health.
Help user with the following request: I am hungry


Now, use OpenAI

In [ ]:
from openai import OpenAI
client = OpenAI(
    base_url="https://bothub.chat/api/v2/openai/v1/", # "http://<Your api-server IP>:port"
    api_key=''
)

In [ ]:
messages = [{"role": "user", "content": new_prompt}]
res = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                max_tokens=100,
                temperature=0.2,

            )
res

ChatCompletion(id='gen-1741713902-fxF0IZ55xQ7QAU5VO5by', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="If you're feeling hungry, you might want to try making or ordering a traditional Italian pizza. It's known for its thin crust and fresh ingredients, and if you have access to a wood-fired oven, it can enhance the flavor even more. Enjoy your meal!", refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None), native_finish_reason='stop')], created=1741713902, model='openai/gpt-4o', object='chat.completion', service_tier=None, system_fingerprint='fp_f9f4fb6dbf', usage=CompletionUsage(completion_tokens=52, prompt_tokens=75, total_tokens=127, completion_tokens_details=None, prompt_tokens_details=None), provider='OpenAI')

In [ ]:
print(res.choices[0].message.content)

If you're feeling hungry, you might want to try making or ordering a traditional Italian pizza. It's known for its thin crust and fresh ingredients, and if you have access to a wood-fired oven, it can enhance the flavor even more. Enjoy your meal!


Preprocessing of the request:

In [ ]:
query = 'Tell me about Leonardo'

In [ ]:
messages = [{"role": "system", "content":
             '''User sends you a query. Think in steps and understand what does they really want to ask.
             Format your answer in the following way:
             Real query: <question>
             Why: <reasoning>

'''},
            {'role':'user', 'content':query}]
res = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                max_tokens=100,
                temperature=0.2,
                #response_format={"type": "json_object"}
            )
res

ChatCompletion(id='gen-1741714180-OKx4THrYKttWrXz7LMWI', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Real query: Who is Leonardo, and what is he known for?\n\nWhy: The user is likely asking for information about a well-known individual named Leonardo. This could refer to several famous people, such as Leonardo da Vinci, the Renaissance artist and inventor, or Leonardo DiCaprio, the contemporary actor. The user wants to know more about this person's identity and achievements.", refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None), native_finish_reason='stop')], created=1741714180, model='openai/gpt-4o', object='chat.completion', service_tier=None, system_fingerprint='fp_eb9dce56a8', usage=CompletionUsage(completion_tokens=75, prompt_tokens=57, total_tokens=132, completion_tokens_details=None, prompt_tokens_details=None), provider='OpenAI')

In [ ]:
print(res.choices[0].message.content)

Real query: Who is Leonardo, and what is he known for?

Why: The user is likely asking for information about a well-known individual named Leonardo. This could refer to several famous people, such as Leonardo da Vinci, the Renaissance artist and inventor, or Leonardo DiCaprio, the contemporary actor. The user wants to know more about this person's identity and achievements.


In [ ]:
preprocessed_query=res.choices[0].message.content.split('\nWhy: ')[0][len('Real query: '):]
preprocessed_query

'Who is Leonardo, and what is he known for?\n'

In [ ]:
relevant_docs = '\n//\n'.join(collection.query(
    query_texts=[preprocessed_query],
    n_results=3,
)['documents'][0])

new_prompt = ('Here are some documents, separated by // :\n' + relevant_docs +
              '\nHelp user with the following request: ' + preprocessed_query)

print(new_prompt)

Here are some documents, separated by // :
Leonardo da Vinci's Mona Lisa is considered one of the most iconic paintings in art history.
//
Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.
//
Einstein's theory of relativity revolutionized our understanding of space and time.
Help user with the following request: Who is Leonardo, and what is he known for?



In [ ]:
client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": new_prompt}],
                max_tokens=100,
                temperature=0.2,
            ).choices[0].message.content

'Leonardo refers to Leonardo da Vinci, who is known for being a highly influential artist and polymath from the Renaissance period. He is most famous for his painting, the Mona Lisa, which is considered one of the most iconic paintings in art history. In addition to his work as a painter, Leonardo da Vinci was also a scientist, engineer, and inventor, contributing significantly to various fields of study.'